# Task 6 — Observable subclasses

Walks through `PauliString`, `PauliSum`, `LocalOp`, and `Magnetization`.  
Focus: physics correctness checks you can verify by eye.

In [1]:
import numpy as np
from qaravan.core.observables import PauliString, PauliSum, LocalOp, Magnetization

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)

## 1. `PauliString` — single and two-qubit matrices

In [2]:
# Single-qubit
for label in ['I','X','Y','Z']:
    ref = {'I': I, 'X': X, 'Y': Y, 'Z': Z}[label]
    ok = np.allclose(PauliString(label).matrix, ref)
    print(f"PauliString('{label}').matrix correct: {ok}")

PauliString('I').matrix correct: True
PauliString('X').matrix correct: True
PauliString('Y').matrix correct: True
PauliString('Z').matrix correct: True


In [3]:
# lower case
# Single-qubit
for label in ['i','x','y','z']:
    ref = {'i': I, 'x': X, 'y': Y, 'z': Z}[label]
    ok = np.allclose(PauliString(label).matrix, ref)
    print(f"PauliString('{label}').matrix correct: {ok}")

PauliString('i').matrix correct: True
PauliString('x').matrix correct: True
PauliString('y').matrix correct: True
PauliString('z').matrix correct: True


In [5]:
PauliString("IXY", coeff=2.0)

((2+0j))*IXY

In [6]:
# Two-qubit: 'XZ' -> kron(X, Z)
ref = np.kron(X, Z)
got = PauliString('XZ').matrix
print('PauliString("XZ") == kron(X,Z):', np.allclose(got, ref))
print(got.real)

PauliString("XZ") == kron(X,Z): True
[[ 0.  0.  1.  0.]
 [ 0. -0.  0. -1.]
 [ 1.  0.  0.  0.]
 [ 0. -1.  0. -0.]]


In [7]:
# Coefficient scaling
ps = PauliString('Z', coeff=0.5)
print('coeff=0.5:', np.allclose(ps.matrix, 0.5 * Z))

# Lowercase accepted
print('lowercase accepted:', np.allclose(PauliString('xz').matrix, np.kron(X, Z)))

coeff=0.5: True
lowercase accepted: True


## 2. Arithmetic: building a Hamiltonian term by term

In [10]:
# Ising-like: -J * ZZ - h * IX
J, h = 1.0, 0.75
obs = (-J) * PauliString('ZZ') + (-h) * PauliString('IX')
print(type(obs))   # should be PauliSum

expected = -J * np.kron(Z, Z) + (-h) * np.kron(I, X)
print('matches direct construction:', np.allclose(obs.matrix, expected))
print()
obs

<class 'qaravan.core.observables.PauliSum'>
matches direct construction: True



((-1+0j))*ZZ + ((-0.75+0j))*IX

In [12]:
# PauliSum + PauliString
full = obs + PauliString('ZI', coeff=0.5)
print(f'terms after adding one more: {len(full.terms)}')
expected2 = expected + 0.5 * np.kron(Z, I)
print('still correct:', np.allclose(full.matrix, expected2))

terms after adding one more: 3
still correct: True


In [13]:
# Scalar multiply of PauliSum
doubled = 2.0 * obs
print('2x scale:', np.allclose(doubled.matrix, 2.0 * obs.matrix))

2x scale: True


## 3. `as_pauli_sum()`

In [14]:
ps = PauliString('XY')
psum = ps.as_pauli_sum()
print('as_pauli_sum() gives PauliSum:', isinstance(psum, PauliSum))
print('matrix unchanged:', np.allclose(psum.matrix, ps.matrix))

# PauliSum.as_pauli_sum() returns self
print('as_pauli_sum() is self for PauliSum:', psum.as_pauli_sum() is psum)

as_pauli_sum() gives PauliSum: True
matrix unchanged: True
as_pauli_sum() is self for PauliSum: True


## 4. `LocalOp` — local matrix, no embedding

In [15]:
op0 = LocalOp(Z, indices=[0])
op1 = LocalOp(Z, indices=[1])

print('LocalOp(Z, [0]).matrix == Z:', np.allclose(op0.matrix, Z))
print('LocalOp(Z, [1]).matrix == Z:', np.allclose(op1.matrix, Z))  # still just Z
print('indices stored correctly:', op0.indices, op1.indices)
print()
print('Note: embedding into full Hilbert space is the backend\'s job.')

LocalOp(Z, [0]).matrix == Z: True
LocalOp(Z, [1]).matrix == Z: True
indices stored correctly: [0] [1]

Note: embedding into full Hilbert space is the backend's job.


In [16]:
# Two-site local op
zz = np.kron(Z, Z)
op_zz = LocalOp(zz, indices=[0, 1])
print('two-site LocalOp.matrix == kron(Z,Z):', np.allclose(op_zz.matrix, zz))
print('indices:', op_zz.indices)

two-site LocalOp.matrix == kron(Z,Z): True
indices: [0, 1]


## 5. `Magnetization` — subclass of `PauliSum`

In [17]:
M2 = Magnetization(2)
print('isinstance(Magnetization(2), PauliSum):', isinstance(M2, PauliSum))
print('terms:', M2.terms)
print()
# Should be 0.5*(IZ + ZI)
expected_M2 = 0.5 * (np.kron(I, Z) + np.kron(Z, I))
print('matrix matches 0.5*(IZ+ZI):', np.allclose(M2.matrix, expected_M2))

isinstance(Magnetization(2), PauliSum): True
terms: [((0.5+0j))*ZI, ((0.5+0j))*IZ]

matrix matches 0.5*(IZ+ZI): True


In [18]:
# Basis-state expectations: <M> should be in [-1, +1]
def expectation_matrix(mat, sv):
    return (sv.conj() @ mat @ sv).real

v00 = np.array([1, 0, 0, 0], dtype=complex)  # |00>
v01 = np.array([0, 1, 0, 0], dtype=complex)  # |01>
v10 = np.array([0, 0, 1, 0], dtype=complex)  # |10>
v11 = np.array([0, 0, 0, 1], dtype=complex)  # |11>

M = M2.matrix
print('<M>|00> =', expectation_matrix(M, v00), ' (expected +1.0)')
print('<M>|01> =', expectation_matrix(M, v01), ' (expected  0.0)')
print('<M>|10> =', expectation_matrix(M, v10), ' (expected  0.0)')
print('<M>|11> =', expectation_matrix(M, v11), ' (expected -1.0)')

<M>|00> = 1.0  (expected +1.0)
<M>|01> = 0.0  (expected  0.0)
<M>|10> = 0.0  (expected  0.0)
<M>|11> = -1.0  (expected -1.0)


In [19]:
# n=4: magnetization diagonal entries
# Computational basis |b3 b2 b1 b0>: M = (1/4) * sum_i (1-2*b_i)
M4 = Magnetization(4)
mat4 = M4.matrix
diag = np.diag(mat4).real

# Expected: for bit string b (MSB first), M = mean(1 - 2*bit)
expected_diag = np.array([
    np.mean([1 - 2*int(b) for b in f'{k:04b}']) for k in range(16)
])
print('Diagonal correct:', np.allclose(diag, expected_diag))
print('Range:', diag.min(), 'to', diag.max(), ' (should be -1 to +1)')

Diagonal correct: True
Range: -1.0 to 1.0  (should be -1 to +1)


In [20]:
# X magnetization: |+>^n should give +1
Mx = Magnetization(2, axis='X')
plus2 = np.array([1, 1, 1, 1], dtype=complex) / 2.0  # |+>|+>
print('<Mx>|++> =', expectation_matrix(Mx.matrix, plus2), ' (expected +1.0)')

<Mx>|++> = 1.0  (expected +1.0)


## 6. Hermiticity check

In [21]:
for name, obs in [
    ('PauliString("XYZ")', PauliString('XYZ')),
    ('PauliSum([ZI, IX])', PauliSum([PauliString('ZI'), PauliString('IX')])),
    ('LocalOp(kron(Z,Z))', LocalOp(np.kron(Z,Z), [0,1])),
    ('Magnetization(3)', Magnetization(3)),
]:
    m = obs.matrix
    herm = np.allclose(m, m.conj().T)
    print(f'{name}: Hermitian = {herm}')

PauliString("XYZ"): Hermitian = True
PauliSum([ZI, IX]): Hermitian = True
LocalOp(kron(Z,Z)): Hermitian = True
Magnetization(3): Hermitian = True
